In [ ]:
import torch
from torch import Tensor
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tiktoken
import math


class WMT14(Dataset):
    def __init__(self, data_split, num_rows):
        super().__init__()
        dataset = load_dataset("wmt/wmt14", "fr-en", split=f"{data_split}[:{num_rows}]")
        self.df = dataset.data.to_pandas()
        self.rows: list[tuple[Tensor, Tensor] | None] = [None for _ in range(num_rows)]
        self.encoder = tiktoken.get_encoding("r50k_base")

    def __len__(self):
        return self.df.shape[0]

    def vocab_size(self):
        return self.encoder.n_vocab

    def tokenize(self, sentence, token_block_size):
        tokens = self.encoder.encode(sentence)
        tokens = tokens[:token_block_size]
        tokens += [0] * abs(token_block_size - len(tokens))
        return torch.tensor(tokens)

    def __getitem__(self, idx):
        if self.rows[idx] is None:
            c = self.df.columns[0]
            self.rows[idx] = (
                self.tokenize(self.df[c].str["en"][idx], 128),
                self.tokenize(self.df[c].str["fr"][idx], 128)
            )
        return self.rows[idx][0], self.rows[idx][1]

In [ ]:
def kaiming_linear(in_size, out_size, bias=False):
    layer = nn.Linear(in_size, out_size, bias=bias)
    nn.init.kaiming_normal_(layer.weight)
    if bias:
        nn.init.zeros_(layer.bias)
    return layer


class TokenEmbedding(nn.Module):
    def __init__(self, tokblk_size, vocab_size, d_model):
        super().__init__()
        # Positional encoding matrix
        div_term = torch.exp((torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)))
        pos = torch.arange(0, tokblk_size).view(-1, 1).expand(tokblk_size, d_model) * div_term

        self.positional = torch.zeros(tokblk_size, d_model)
        self.positional[:, 0::2] = torch.sin(pos[:, 0::2])
        self.positional[:, 1::2] = torch.cos(pos[:, 1::2])

        # Let Pytorch know that this isn't a learned parameter
        self.register_buffer("positional", self.positional)

        # Embedding matrix
        self.weights = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # Map each token (scalar) to the corresponding embedding vector
        # x.shape == (batch_size, tokblk_size), self.weights.shape == (tokblk_size, d_model),
        # result.shape == (batch_size, tokblk_size, d_model)
        return self.weights(x) + self.positional


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, h):
        super().__init__()
        self.W_q = kaiming_linear(d_model, h * d_k)
        self.W_k = kaiming_linear(d_model, h * d_k)
        self.W_v = kaiming_linear(d_model, h * d_v)
        self.W_o = kaiming_linear(d_model, h * d_v)
        self.h, self.d_k = h, d_k

    def forward(self, Q, K, V, mask=False):
        # Project Q, K, V into smaller subspaces using each h projection matrices
        B, T = Q.shape[0], Q.shape[1],
        Q_proj = self.W_q(Q).view(B, T, self.h, self.d_k).transpose(1, 2)
        K_proj = self.W_k(K).view(B, T, self.h, self.d_k).transpose(1, 2)
        V_proj = self.W_v(V).view(B, T, self.h, self.d_k).transpose(1, 2)

        # Compute mask used for masked self attention, ensuring that
        # each token in the sequence only attends to prior tokens
        M = 0
        if mask:
            B, num_tokens = Q_proj.shape[0], Q_proj.shape[2]
            inf_matrix = torch.full((B, num_tokens, num_tokens), -torch.inf)
            M = torch.triu(inf_matrix, diagonal=1)

        # Compute attention for each attention head:
        # Q @ K.T for each attention head in each batch to get similarity matrices
        logits = torch.einsum("bhij,bhkj->bhik", Q_proj, K_proj)
        scores = torch.softmax((logits + M) / Q_proj.shape[-1], dim=-1)
        scaled = torch.einsum("bhij,bhjk->bhik", scores, V_proj)

        # Concatenate the scaled values for each attention head together and
        # project the resulting tensor into the original vector space
        concat = torch.permute(scaled, (0, 2, 1, 3)).reshape(B, T, self.h * self.d_v)
        return self.W_o(concat).view(B, T, self.h * self.d_k)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_model // h, d_model // h, h)
        self.norm1 = nn.LayerNorm(d_model)
        self.l1 = kaiming_linear(d_model, d_model * 4, bias=True)
        self.relu = nn.ReLU()
        self.l2 = kaiming_linear(d_model * 4, d_model, bias=True)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Self-attention, residual connection and LayerNorm
        attn_out = self.attn(x, x, x)
        attn_out = self.norm1(attn_out + x)
        # Feed forward network, residual connection and LayerNorm
        ff_out = self.l1(attn_out)
        ff_out = self.relu(ff_out)
        ff_out = self.l2(ff_out)
        return self.norm2(ff_out + attn_out)


class Decoder(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.attn1 = MultiHeadAttention(d_model, d_model // h, d_model // h, h)
        self.norm1 = nn.LayerNorm(d_model)
        self.attn2 = MultiHeadAttention(d_model, d_model // h, d_model // h, h)
        self.norm2 = nn.LayerNorm(d_model)
        self.l1 = kaiming_linear(d_model, d_model * 4, bias=True)
        self.relu = nn.ReLU()
        self.l2 = kaiming_linear(d_model * 4, d_model, bias=True)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, Q):
        # Masked self-attention, residual connection and LayerNorm
        attn_out1 = self.attn1(x, x, x, True)
        attn_out1 = self.norm1(attn_out1 + x)
        # Cross self-attention, residual connection and LayerNorm
        attn_out2 = self.attn2(Q, attn_out1, attn_out1)
        attn_out2 = self.norm2(attn_out2 + attn_out1)
        # Feed forward network, residual connection and LayerNorm
        ff_out = self.l1(attn_out2)
        ff_out = self.relu(ff_out)
        ff_out = self.l2(ff_out)
        return self.norm3(ff_out + attn_out2)


class Transformer(nn.Module):
    def __init__(self, tokblk_size, vocab_size, d_model, h, N):
        super().__init__()
        self.tokemb = TokenEmbedding(tokblk_size, vocab_size, d_model)
        self.encoder_layers = nn.ModuleList([Encoder(d_model, h) for _ in range(N)])
        self.decoder_layers = nn.ModuleList([Decoder(d_model, h) for _ in range(N)])
        self.token_out = kaiming_linear(d_model, vocab_size, bias=True)

    def forward(self, x_in, x_out):
        in_embeddings = self.tokemb(x_in)
        for layer in self.encoder_layers:
            in_embeddings = layer(in_embeddings)

        # Shift the output sequence to the right so that the model
        # doesn't learn to simply repeat the last input token.
        shifted = torch.roll(x_out, shifts=1, dims=1)
        out_embeddings = self.tokemb(shifted)
        for layer in self.decoder_layers:
            # Q for cross self-attention is the final output of the encoder
            out_embeddings = layer(out_embeddings, in_embeddings)

        # Output probability distribution shape: (B, num tokens, vocabulary size)
        return self.token_out(out_embeddings)

In [ ]:
# The target training task is machine translation. Each training step, input english sequences
# and the french output sequences are tokenized, padded and batched before being fed into the
# Transformer. The Transformer outputs a probability distribution of what each output sequence
# token could be, and during training cross-entropy loss is used to see how much they differ.
def run_model(model, criterion, loader, optimizer, device, training):
    if training:
        model.train()
    else:
        model.eval()

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for en_batch, fr_batch in loader:
            en_batch, fr_batch = en_batch.to(device), fr_batch.to(device)

            prediction = model(en_batch, fr_batch)
            print(prediction.shape)
            loss = criterion(prediction, fr_batch) # TODO: do this properly and plot error and loss

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

warmup_steps, train_steps, d_model = 1, 10, 512
lr_lambda = lambda step: (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))

train_dataset = WMT14("train", 100)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

model = Transformer(128, train_dataset.vocab_size(), d_model, 8, 6).to(device)
optimizer = Adam(model.parameters(), lr=lr_lambda(1), betas=(0.9, 0.98), eps=1e-9)
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
criterion = nn.CrossEntropyLoss()

for i in range(warmup_steps + train_steps):
    run_model(model, criterion, train_loader, optimizer, device, True)
    print(f"Step {i}/{warmup_steps + train_steps}")